# Dar_028 — Existing-data environmental comparison

Use the **real pinned EDA already in this project** to compare the current 21 environmental components. No new downloads, NASA mission profile or synthetic candidates are required for the main run.

This is the rule-based environmental companion to the terrain CNN. The current components are not yet matched to the CNN's patch identities. Context rankings are not operational Stage 3 eligibility or independent validation.


In [1]:
from pathlib import Path
import sys, json
sys.dont_write_bytecode = True
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'outline.md').is_file() and (p/'notebooks').is_dir())
LOCAL = ROOT/'darrel stuff'
SOURCE = LOCAL/'Pipeline_Proto'
DATA = LOCAL/'data'
OUTPUTS = SOURCE/'outputs'
sys.path.insert(0, str(SOURCE))
import pandas as pd
from IPython.display import display
import environment_engine as engine
CONFIG = SOURCE/'environment_experiment.json'
POLICY = SOURCE/'diagnostic_policy.json'
config = json.loads(CONFIG.read_text(encoding='utf-8'))
display(pd.DataFrame(config['sources'].items(), columns=['evidence_package', 'pinned_analysis_run']))


,evidence_package,pinned_analysis_run
0,candidates,processed/analysis/dar_020_candidate_environme...
1,shadowcam,processed/analysis/dar_026_shadowcam_manual/20...
2,minirf,processed/analysis/dar_027_minirf_manual/20261...


## 1. Run the available-data product

The normal run uses only the verified existing EDA and the declared context-comparison policy. It does not load a missing mission/equipment profile or generate operational-readiness tables. Unsupported coverage stays counted, and excluded candidates stay accounted for; no values are invented.


In [2]:
comparison_run = engine.main(['--config', str(CONFIG)])
assert comparison_run.is_relative_to(OUTPUTS) and not comparison_run.is_relative_to(DATA)
status = pd.read_csv(comparison_run/'candidate_status.csv')
evidence = pd.read_csv(comparison_run/'environment_evidence.csv')
ranking = pd.read_csv(comparison_run/'environmental_rankings.csv')
top = pd.read_csv(comparison_run/'environmental_top_k.csv')
report = json.loads((comparison_run/'report.json').read_text(encoding='utf-8'))
display(evidence.groupby('metric').size().rename('supported_candidates').reset_index())
assert len(status) == 21 and len(ranking) == 14 and len(top) == 5
assert status.comparison_status.value_counts().to_dict() == {'ranked_context_only': 14, 'excluded_by_context_filter': 7}
assert evidence.status.eq('observed').all() and not evidence.isna().any().any()
assert ranking.environmental_score.notna().all()
assert report['product_mode'] and not report['is_synthetic']
assert not report['stage3_assessment_requested']


Existing-data environmental comparison (not mission/equipment adequacy)
{
  "run_kind": "environment_comparison",
  "data_basis": "pinned_measured_eda_with_qualified_context",
  "candidates": 21,
  "primary_result": "environmental_rankings.csv",
  "environmental_comparison_statuses": {
    "ranked_context_only": 14,
    "excluded_by_context_filter": 7
  },
  "environmental_top_k_returned": 5,
  "comparison_policy_file": "darrel stuff\\Pipeline_Proto\\diagnostic_policy.json"
}
Equipment/mission assessment not requested. Use --stage3 or --mission-profile only when needed.
Outputs: C:\Darrel\Self-Projects\1 group proj\NASA-Space-Apps-Challenge-2026\darrel stuff\Pipeline_Proto\outputs\environment_comparison\20261109T184313460859Z_732b52
Readable summary: C:\Darrel\Self-Projects\1 group proj\NASA-Space-Apps-Challenge-2026\darrel stuff\Pipeline_Proto\outputs\environment_comparison\20261109T184313460859Z_732b52\RUN_SUMMARY.md
Start here: C:\Darrel\Self-Projects\1 group proj\NASA-Space-Apps-Ch

,metric,supported_candidates
0,earth_visibility_median,21
1,minirf_cpr_median,19
2,shadowcam_radiance_median,3
3,slope_q90_deg,21
4,solar_visibility_median,21
5,thermal_phase_median_max_K,5
6,thermal_phase_median_min_K,5
7,thermal_phase_median_span_K,5


## 2. Rankings and the configured factors

Reuse the existing policy: sunlight and Earth visibility receive equal preference weights, while slope is a context filter, not another score weight. These settings are **illustrative comparison criteria**, not equipment or safety limits. The components were already selected using related proxies, so the ranking is not independent validation.


In [3]:
policy = json.loads(POLICY.read_text(encoding='utf-8'))
display(pd.DataFrame(policy['soft_preferences'])[['id','metric','lower','upper','weight','reason']])
display(top[['candidate_id','region','solar_visibility_median','earth_visibility_median','slope_q90_deg','environmental_score','environmental_rank']])
assert top.context_filter_status.eq('pass').all()
assert top.environmental_score.between(0,1).all()
assert not top.candidate_id.str.startswith('SYN_').any()


,id,metric,lower,upper,weight,reason
0,sunlight_context,solar_visibility_median,0.0,1.0,1.0,Equal-weight diagnostic map-visibility prefere...
1,earth_context,earth_visibility_median,0.0,1.0,1.0,Separate illustrative contact context; not lin...


,candidate_id,region,solar_visibility_median,earth_visibility_median,slope_q90_deg,environmental_score,environmental_rank
0,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,A3CLR22_10_Leibnitz_Beta_Plateau,0.51422,0.99960,7.842590,0.75691,1
1,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,A3CLR22_10_Leibnitz_Beta_Plateau,0.52364,0.81464,6.919683,0.66914,2
2,A3CLR22_2_Haworth:diagnostic_component_00014,A3CLR22_2_Haworth,0.53240,0.67404,7.864025,0.60322,3
3,A3CLR22_4_deGerlache_Rim_2:diagnostic_componen...,A3CLR22_4_deGerlache_Rim_2,0.58784,0.61646,7.889862,0.60215,4
4,A3CLR22_5_Connecting_Ridge_Extension:diagnosti...,A3CLR22_5_Connecting_Ridge_Extension,0.59540,0.58824,5.747799,0.59182,5


## 3. Real scores and real exclusions

`environmental_rankings.csv` contains only candidates with complete, comparable scores. `candidate_status.csv` accounts for all inputs, while `excluded_candidates.csv` explains the rest without empty score cells. A missing factor is still a genuine evidence gap—not a zero, a reweighted score or an implicit pass.


In [4]:
display(status.groupby('comparison_status').size().rename('candidates').reset_index())
excluded = pd.read_csv(comparison_run/'excluded_candidates.csv')
display(excluded)
checks = pd.read_csv(comparison_run/'hard_rule_results.csv')
display(checks.loc[checks.status.eq('fail'), ['candidate_id','rule_id','value','operator','limit','unit','scope']])
utilities = pd.read_csv(comparison_run/'soft_preference_results.csv')
display(utilities.loc[utilities.candidate_id.isin(top.candidate_id), ['candidate_id','preference_id','value','weight','utility']])
assert not set(ranking.candidate_id).intersection(excluded.candidate_id)
assert set(status.candidate_id) == set(ranking.candidate_id) | set(excluded.candidate_id)
assert set(utilities.candidate_id) <= set(ranking.candidate_id)
assert utilities.utility.notna().all()


,comparison_status,candidates
0,excluded_by_context_filter,7
1,ranked_context_only,14


,candidate_id,region,comparison_status,reason
0,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,A3CLR22_10_Leibnitz_Beta_Plateau,excluded_by_context_filter,illustrative_slope: slope_q90_deg=8.7972885131...
1,A3CLR22_11_Malapert_Massif:diagnostic_componen...,A3CLR22_11_Malapert_Massif,excluded_by_context_filter,illustrative_slope: slope_q90_deg=9.1490782737...
2,A3CLR22_11_Malapert_Massif:diagnostic_componen...,A3CLR22_11_Malapert_Massif,excluded_by_context_filter,illustrative_slope: slope_q90_deg=8.1597366333...
3,A3CLR22_11_Malapert_Massif:diagnostic_componen...,A3CLR22_11_Malapert_Massif,excluded_by_context_filter,illustrative_slope: slope_q90_deg=8.2072187423...
4,A3CLR22_12_deGerlache_Kocher_Massif:diagnostic...,A3CLR22_12_deGerlache_Kocher_Massif,excluded_by_context_filter,illustrative_slope: slope_q90_deg=8.2895782470...
5,A3CLR22_12_deGerlache_Kocher_Massif:diagnostic...,A3CLR22_12_deGerlache_Kocher_Massif,excluded_by_context_filter,illustrative_slope: slope_q90_deg=8.3011487960...
6,A3CLR22_2_Haworth:diagnostic_component_00008,A3CLR22_2_Haworth,excluded_by_context_filter,illustrative_slope: slope_q90_deg=8.9693218231...


,candidate_id,rule_id,value,operator,limit,unit,scope
5,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,illustrative_slope,8.797289,le,8.0,degree,static_map_cell_summary
11,A3CLR22_11_Malapert_Massif:diagnostic_componen...,illustrative_slope,8.207219,le,8.0,degree,static_map_cell_summary
14,A3CLR22_11_Malapert_Massif:diagnostic_componen...,illustrative_slope,9.149078,le,8.0,degree,static_map_cell_summary
17,A3CLR22_11_Malapert_Massif:diagnostic_componen...,illustrative_slope,8.159737,le,8.0,degree,static_map_cell_summary
20,A3CLR22_12_deGerlache_Kocher_Massif:diagnostic...,illustrative_slope,8.301149,le,8.0,degree,static_map_cell_summary
23,A3CLR22_12_deGerlache_Kocher_Massif:diagnostic...,illustrative_slope,8.289578,le,8.0,degree,static_map_cell_summary
32,A3CLR22_2_Haworth:diagnostic_component_00008,illustrative_slope,8.969322,le,8.0,degree,static_map_cell_summary


,candidate_id,preference_id,value,weight,utility
0,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,sunlight_context,0.52364,1.0,0.52364
1,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,earth_context,0.81464,1.0,0.81464
2,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,sunlight_context,0.51422,1.0,0.51422
3,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,earth_context,0.99960,1.0,0.99960
6,A3CLR22_2_Haworth:diagnostic_component_00014,sunlight_context,0.53240,1.0,0.53240
7,A3CLR22_2_Haworth:diagnostic_component_00014,earth_context,0.67404,1.0,0.67404
20,A3CLR22_4_deGerlache_Rim_2:diagnostic_componen...,sunlight_context,0.58784,1.0,0.58784
21,A3CLR22_4_deGerlache_Rim_2:diagnostic_componen...,earth_context,0.61646,1.0,0.61646
24,A3CLR22_5_Connecting_Ridge_Extension:diagnosti...,sunlight_context,0.59540,1.0,0.59540
25,A3CLR22_5_Connecting_Ridge_Extension:diagnosti...,earth_context,0.58824,1.0,0.58824


## 4. Use the other supported environmental evidence

All 21 examples have qualifying sunlight, Earth visibility and slope. Five have complete common-phase thermal profiles. Qualified ShadowCam/Mini-RF measurements are retained in the available evidence, without pretending they measure local equipment or ground capacity.

Only supported values are exported. Coverage counts identify omitted measurements. Complete temperature profiles remain descriptive phase-point evidence, not mission-time extrema or component response. Full mission/equipment qualification is not run unless explicitly requested with `--stage3` or `--mission-profile`.


In [5]:
usage = pd.read_csv(comparison_run/'feature_usage.csv')
display(usage)
assert set(usage.loc[usage.used_in_score,'metric']) == {'solar_visibility_median','earth_visibility_median'}
assert not usage.set_index('metric').loc['slope_q90_deg','used_in_score']
if (comparison_run/'thermal_profiles.csv').exists():
    display(pd.read_csv(comparison_run/'thermal_profiles.csv'))
display(evidence.loc[evidence.metric.isin(['shadowcam_radiance_median','minirf_cpr_median'])])
assert not (comparison_run/'stage3_operational_checks.csv').exists()
assert not (comparison_run/'stage3_top_k.csv').exists()
assert not report['operational_suitability_assessed']


,metric,unit,scope,candidate_count,supported_candidates,unsupported_or_absent_candidates,used_in_context_filter,used_in_score,reason
0,earth_visibility_median,fraction,static_map_cell_summary,21,21,0,True,True,Used only in the explicitly configured context...
1,minirf_cpr_median,dimensionless,associated_native_centres_context_only,21,19,2,False,False,Context only: source/support contract forbids ...
2,shadowcam_radiance_median,W m^-2 sr^-1 um^-1,associated_native_centres_context_only,21,3,18,False,False,Context only: source/support contract forbids ...
3,slope_q90_deg,degree,static_map_cell_summary,21,21,0,True,False,Used only in the explicitly configured context...
4,solar_visibility_median,fraction,static_map_cell_summary,21,21,0,True,True,Used only in the explicitly configured context...
5,thermal_phase_median_max_K,K,fixed_common_phase_point_context,21,5,16,False,False,Retained as context; not in the declared score...
6,thermal_phase_median_min_K,K,fixed_common_phase_point_context,21,5,16,False,False,Retained as context; not in the declared score...
7,thermal_phase_median_span_K,K,fixed_common_phase_point_context,21,5,16,False,False,Retained as context; not in the declared score...


,candidate_id,thermal_phase_median_min_K,thermal_phase_median_max_K,thermal_phase_median_span_K,region,scope
0,A3CLR22_11_Malapert_Massif:diagnostic_componen...,69.575676,246.704788,177.129112,A3CLR22_11_Malapert_Massif,fixed_common_phase_point_context_not_component...
1,A3CLR22_11_Malapert_Massif:diagnostic_componen...,67.003153,237.080544,170.077390,A3CLR22_11_Malapert_Massif,fixed_common_phase_point_context_not_component...
2,A3CLR22_11_Malapert_Massif:diagnostic_componen...,66.875706,223.188446,156.312740,A3CLR22_11_Malapert_Massif,fixed_common_phase_point_context_not_component...
3,A3CLR22_2_Haworth:diagnostic_component_00008,68.863293,249.140403,180.277111,A3CLR22_2_Haworth,fixed_common_phase_point_context_not_component...
4,A3CLR22_2_Haworth:diagnostic_component_00014,64.288193,234.181763,169.893570,A3CLR22_2_Haworth,fixed_common_phase_point_context_not_component...


,candidate_id,metric,value,unit,scope,support_count,status,source_run
78,A3CLR22_4_deGerlache_Rim_2:diagnostic_componen...,shadowcam_radiance_median,0.360309,W m^-2 sr^-1 um^-1,associated_native_centres_context_only,2,observed,processed/analysis/dar_026_shadowcam_manual/20...
79,A3CLR22_4_deGerlache_Rim_2:diagnostic_componen...,shadowcam_radiance_median,0.314223,W m^-2 sr^-1 um^-1,associated_native_centres_context_only,2,observed,processed/analysis/dar_026_shadowcam_manual/20...
80,A3CLR22_4_deGerlache_Rim_2:diagnostic_componen...,shadowcam_radiance_median,0.080437,W m^-2 sr^-1 um^-1,associated_native_centres_context_only,3,observed,processed/analysis/dar_026_shadowcam_manual/20...
81,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,minirf_cpr_median,0.408353,dimensionless,associated_native_centres_context_only,12,observed,processed/analysis/dar_027_minirf_manual/20261...
82,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,minirf_cpr_median,0.769147,dimensionless,associated_native_centres_context_only,1,observed,processed/analysis/dar_027_minirf_manual/20261...
83,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,minirf_cpr_median,0.913352,dimensionless,associated_native_centres_context_only,2,observed,processed/analysis/dar_027_minirf_manual/20261...
84,A3CLR22_11_Malapert_Massif:diagnostic_componen...,minirf_cpr_median,0.368116,dimensionless,associated_native_centres_context_only,12,observed,processed/analysis/dar_027_minirf_manual/20261...
85,A3CLR22_11_Malapert_Massif:diagnostic_componen...,minirf_cpr_median,0.402620,dimensionless,associated_native_centres_context_only,11,observed,processed/analysis/dar_027_minirf_manual/20261...
86,A3CLR22_11_Malapert_Massif:diagnostic_componen...,minirf_cpr_median,1.191691,dimensionless,associated_native_centres_context_only,4,observed,processed/analysis/dar_027_minirf_manual/20261...
87,A3CLR22_12_deGerlache_Kocher_Massif:diagnostic...,minirf_cpr_median,0.738279,dimensionless,associated_native_centres_context_only,3,observed,processed/analysis/dar_027_minirf_manual/20261...


## 4b. Evidence-backed rules and the Step 3 findings

The [Step 4 rule register](../research/environment_rule_register.json) separates conditional operational requirements, historical references, evidence-use guards and the actual exploratory comparison settings. New Dar_033-039 evidence is loaded from pinned, hash-checked outputs; Dar_037/038 history is retained through the accepted Dar_039 extension, not counted twice.

A **Failed evidence-use check** rejects an unsupported inference; it does not declare the physical site unsafe. Orbital flux/counters are not crew dose; zero strict dated thermal observations is not a complete temperature history; finite ROI-center geometry is not a validated component power/contact history. Mission/equipment inputs remain optional. No new factor/weight or operational pass is added to the comparison.

In [6]:
from IPython.display import Markdown
report = json.loads((comparison_run/'report.json').read_text(encoding='utf-8'))
assert report['evidence_rules']['new_scored_factors'] == 0
assert report['evidence_rules']['new_operational_passes'] == 0
annex = comparison_run/'evidence_context'
register = pd.read_csv(annex/'rule_register.csv')
use_checks = pd.read_csv(annex/'evidence_use_checks.csv')
findings = pd.read_csv(annex/'evidence_findings.csv')
limits = pd.read_csv(annex/'candidate_inference_limits.csv')
assert use_checks.assessment_domain.eq('evidence_use_NOT_site_safety').all()
assert use_checks.candidate_operational_conclusion_from_these_sources.eq('Unknown').all()
assert set(limits.candidate_id) == set(status.candidate_id)
assert limits.candidate_specific_conclusion.eq('Unknown').all()
display(register[['id','role','research_ids','source','unit','limit','lower','upper','weight']])
display(use_checks[['rule_id','check_id','data_use_status','reason','consequence']])
display(findings)
explained = pd.read_csv(annex/'explained_context_checks.csv')
display(explained[['candidate_id','metric','value','operator','limit','unit','status','comparison_filter_distance','distance_kind']].head(12))
geometry_reference = pd.read_csv(annex/'sources/geometry/reference_observers.csv')
assert 'candidate_id' not in geometry_reference
display(geometry_reference)
relative = (annex/'EVIDENCE_DECISIONS.md').relative_to(SOURCE).as_posix()
display(Markdown(f"**[Open all per-candidate conditions, checks and limitations]({relative})**"))


,id,role,research_ids,source,unit,limit,lower,upper,weight
0,E01,evidence_guard,D01-D38,environment_engine.validate_evidence; outline ...,requirement_specific,not_adopted,not_applicable,not_applicable,not_applicable
1,E02,evidence_guard,D01 D20 M01,Dar_030/035; Research_Notes.md Temperature & T...,requirement_specific,not_adopted,not_applicable,not_applicable,not_applicable
2,E03,evidence_guard,D20,Dar_035 strict zero-flag cohort; conservative ...,requirement_specific,not_adopted,not_applicable,not_applicable,not_applicable
3,E04,evidence_guard,D07 D08 D31 M06,Dar_034/037-039; Research_Notes.md Radiation,requirement_specific,not_adopted,not_applicable,not_applicable,not_applicable
4,E05,evidence_guard,D02 D03 D19 M07 M08,Dar_036; outline 2.2B; PGDA 69/90,requirement_specific,not_adopted,not_applicable,not_applicable,not_applicable
5,E06,evidence_guard,M01 M02 M04 M07 M14,Dar_033; Apollo 17 primary report Table 9-II a...,requirement_specific,not_adopted,not_applicable,not_applicable,not_applicable
6,E07,evidence_guard,M04 M14,outline Stage 3; stage3_assessment; existing r...,requirement_specific,not_adopted,not_applicable,not_applicable,not_applicable
7,E08,evidence_guard,D13 D14 D33 M13,Dar_007/008/034; Research_Notes.md Seismic Act...,requirement_specific,not_adopted,not_applicable,not_applicable,not_applicable
8,E09,evidence_guard,D05 D06 D16 D29 D30 D34 M05 M09,outline 2.2D; Dar_016/026/027/031,requirement_specific,not_adopted,not_applicable,not_applicable,not_applicable
9,H_THERMAL,conditional_requirement,D01 D04 D20 D21 D28 D38 M01,Research_Notes.md P1 Temperature & Thermal; ou...,requirement_specific,not_adopted,not_applicable,not_applicable,not_applicable


,rule_id,check_id,data_use_status,reason,consequence
0,E03,qualified_dated_thermal_history,Failed,No strict observations in this frozen cohort,Withhold candidate thermal adequacy; preserve ...
1,E02,surface_phase_to_component_response,Failed,No component thermal-response measurement or a...,"Keep existing phase profiles as context, not e..."
2,E04,orbital_data_to_shielded_dose,Failed,Orbital quantities and published scenarios are...,Do not infer a radiation safety threshold or c...
3,E05,reference_point_to_component_history,Failed,"Finite range, declared 2020 windows and refere...",Retain regional reference coordinates and sens...
4,E06,historical_reserve_arithmetic,Passed,Remaining minus source redline reproduces acce...,Historical comparison only; zero at source rou...
5,E06,historical_limit_to_new_configuration,Failed,No applicable modern mission/equipment identit...,"Do not adopt historical redlines, stocks or cr..."
6,E08,catalog_to_local_event_probability,Failed,"Independence, observation effort, dates and lo...","Keep seismic/impact concerns unresolved, not z..."
7,E09,indirect_indicator_to_resources_or_strength,Failed,"No verified local extractability, stocks, bear...",Do not credit essential supplies or load/mobil...
8,E10,radiation_coverage_to_continuous_mission_history,Failed,File span and incomplete quality/state coverag...,Retain gaps/flags and withhold mission-specifi...
9,E11,radiation_instrument_identity_preserved,Passed,Daily platform identity and summed bins agree;...,"Evidence-handling pass only, not hardware prot..."


,finding,value,unit,scope,decision_implication,source_id
0,actual_consumption_above_prediction,1.600000e+01,of 18 historical records,Apollo 17 EMU Table 9-II only,Prediction is not guaranteed adequacy; no mode...,apollo/primary_resource_changes.csv
1,minimum_historical_headroom::Battery charge,8.200000e-01,Ah,historical source remaining minus redline,Not an actual configured mission margin,apollo/primary_resource_changes.csv
2,minimum_historical_headroom::Feedwater,5.000000e-02,lb,historical source remaining minus redline,Not an actual configured mission margin,apollo/primary_resource_changes.csv
3,minimum_historical_headroom::Oxygen,0.000000e+00,lb,historical source remaining minus redline,Not an actual configured mission margin,apollo/primary_resource_changes.csv
4,dated_thermal_strict_observations,0.000000e+00,channel query rows,frozen query cohort; preserve region/channel/m...,No qualified mission temperature series inferred,dated_thermal/dated_thermal_support.csv
5,dated_thermal_query_products,8.120000e+02,query products,frozen query cohort; preserve region/channel/m...,More downloaded queries cannot compensate for ...,dated_thermal/dated_thermal_support.csv
6,CRaTER_completed_dates,4.940000e+02,UTC dates,historical baseline cohort: 2023-07-01 to 2024...,Date span is not usable exposure coverage or a...,annual_radiation
7,GOES16_completed_dates,4.950000e+02,UTC dates,historical baseline cohort: 2023-11-01 to 2025...,Date span is not usable exposure coverage or a...,annual_radiation
8,CRaTER_SEC_records,4.263981e+07,SEC records,orbital counter context,Counters are not surface/shielded crew dose,annual_radiation/refreshed_crater_daily_contex...
9,GOES_qualified_channel_time_fraction,1.040722e-01,fraction,two directions and thirteen separate energy bands,No filling unqualified intervals; no crew-dose...,annual_radiation/refreshed_goes_daily_directio...


,candidate_id,metric,value,operator,limit,unit,status,comparison_filter_distance,distance_kind
0,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,solar_visibility_median,0.523640,ge,0.5,fraction,pass,0.023640,exploratory_threshold_distance_NOT_operating_m...
1,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,earth_visibility_median,0.814640,ge,0.5,fraction,pass,0.314640,exploratory_threshold_distance_NOT_operating_m...
2,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,slope_q90_deg,6.919683,le,8.0,degree,pass,1.080317,exploratory_threshold_distance_NOT_operating_m...
3,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,solar_visibility_median,0.542600,ge,0.5,fraction,pass,0.042600,exploratory_threshold_distance_NOT_operating_m...
4,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,earth_visibility_median,0.866560,ge,0.5,fraction,pass,0.366560,exploratory_threshold_distance_NOT_operating_m...
5,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,slope_q90_deg,8.797289,le,8.0,degree,fail,-0.797289,exploratory_threshold_distance_NOT_operating_m...
6,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,solar_visibility_median,0.514220,ge,0.5,fraction,pass,0.014220,exploratory_threshold_distance_NOT_operating_m...
7,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,earth_visibility_median,0.999600,ge,0.5,fraction,pass,0.499600,exploratory_threshold_distance_NOT_operating_m...
8,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,slope_q90_deg,7.842590,le,8.0,degree,pass,0.157410,exploratory_threshold_distance_NOT_operating_m...
9,A3CLR22_11_Malapert_Massif:diagnostic_componen...,solar_visibility_median,0.534660,ge,0.5,fraction,pass,0.034660,exploratory_threshold_distance_NOT_operating_m...


,region,x_m,y_m,longitude_deg,latitude_deg,surface_elevation_m,selection
0,A3CLR22_10_Leibnitz_Beta_Plateau,73020.0,117980.0,31.754130,-85.426791,6397.975586,"nearest native cell to ROI center, not recomme..."
1,A3CLR22_11_Malapert_Massif,-500.0,121500.0,-0.235784,-85.994785,4784.007812,"nearest native cell to ROI center, not recomme..."
2,A3CLR22_12_deGerlache_Kocher_Massif,-113380.0,-56100.0,-116.326033,-85.830141,2705.069092,"nearest native cell to ROI center, not recomme..."
3,A3CLR22_2_Haworth,-37980.0,90500.0,-22.766259,-86.764197,1626.623169,"nearest native cell to ROI center, not recomme..."
4,A3CLR22_3_Faustini_Rim_A,64020.0,-20.0,90.017899,-87.888994,48.404945,"nearest native cell to ROI center, not recomme..."
5,A3CLR22_4_deGerlache_Rim_2,-48500.0,22980.0,-64.647730,-88.230260,-86.969101,"nearest native cell to ROI center, not recomme..."
6,A3CLR22_5_Connecting_Ridge_Extension,-28700.0,-6100.0,-101.999287,-89.032416,1178.329224,"nearest native cell to ROI center, not recomme..."


**[Open all per-candidate conditions, checks and limitations](outputs/environment_comparison/20261109T184313460859Z_732b52/evidence_context/EVIDENCE_DECISIONS.md)**

## 5. Optional synthetic software test — OFF by default

The main comparison above uses only existing measured EDA. The ten-row fixture is retained solely as an optional check of the margin functions. It is not required, scored, or substituted into the real-data run. Change the explicit flag below only if you want to run that separate test.


## Science-based inference from our EDA

The original visibility score is a baseline. The science view compares geometric radiation openness, UV opportunity and Earth visibility without arbitrary hazard weights. A second view requires complete thermal cohorts. Plasma remains sourced charging context, not a lowest-density-is-safest score. Relative fronts/ranks are not operational approval.

In [7]:
science = report.get("evidence_rules", {}).get("science_models")
if science:
    science_root = comparison_run / "evidence_context/science"
    fronts = pd.read_csv(science_root / "science_preference_fronts.csv")
    physical = pd.read_csv(science_root / "science_operational_requirements.csv")
    assert physical.status.eq("Unknown").all()
    assert fronts.operational_suitability.eq("Unknown").all()
    assert fronts.loc[fronts.preference_status.ne("compared_context_only"), "pareto_front"].isna().all()
    display(fronts.groupby(["view", "preference_status"]).size().rename("candidates").reset_index())
    display(fronts.loc[fronts.pareto_front.eq(1)])
    display(pd.read_csv(science_root / "science_preference_details.csv").head(12))
    display(pd.read_csv(science_root / "candidate_plasma_context.csv").head())
    display(Markdown("**[Open science comparison, reference assumptions and options](" + (science_root / "SCIENCE_COMPARISON.md").relative_to(SOURCE).as_posix() + ")**"))
else:
    print("This configuration does not enable the optional science extension.")

,view,preference_status,candidates
0,environmental_exposure_tradeoff,compared_context_only,14
1,environmental_exposure_tradeoff,excluded_by_current_context_filters,7
2,with_thermal_stability,Unknown,13
3,with_thermal_stability,excluded_by_current_context_filters,7
4,with_thermal_stability,insufficient_comparable_candidates,1


,candidate_id,region,view,preference_status,pareto_front,baseline_visibility_score,sky_visible_fraction_4pi,uv_sun_visibility_opportunity_proxy,earth_visibility_fraction_median,reason,weighting,operational_suitability,thermal_phase_median_span_K
2,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,A3CLR22_10_Leibnitz_Beta_Plateau,environmental_exposure_tradeoff,compared_context_only,1.0,0.75691,0.501402,0.51422,0.99960,Pareto tradeoff among candidates passing the c...,none_Pareto_not_a_composite_risk_score,Unknown,NaN
16,A3CLR22_4_deGerlache_Rim_2:diagnostic_componen...,A3CLR22_4_deGerlache_Rim_2,environmental_exposure_tradeoff,compared_context_only,1.0,0.55731,0.492688,0.53104,0.58358,Pareto tradeoff among candidates passing the c...,none_Pareto_not_a_composite_risk_score,Unknown,NaN


,view,candidate_id,metric,value,direction,relative_preference_rank,unit,basis,meaning
0,environmental_exposure_tradeoff,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,sky_visible_fraction_4pi,0.508102,lower,14,fraction of 4*pi solid angle,Modelled,Relative position within this comparable cohor...
1,environmental_exposure_tradeoff,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,sky_visible_fraction_4pi,0.501402,lower,3,fraction of 4*pi solid angle,Modelled,Relative position within this comparable cohor...
2,environmental_exposure_tradeoff,A3CLR22_12_deGerlache_Kocher_Massif:diagnostic...,sky_visible_fraction_4pi,0.503328,lower,5,fraction of 4*pi solid angle,Modelled,Relative position within this comparable cohor...
3,environmental_exposure_tradeoff,A3CLR22_2_Haworth:diagnostic_component_00014,sky_visible_fraction_4pi,0.504967,lower,8,fraction of 4*pi solid angle,Modelled,Relative position within this comparable cohor...
4,environmental_exposure_tradeoff,A3CLR22_2_Haworth:diagnostic_component_00069,sky_visible_fraction_4pi,0.504354,lower,6,fraction of 4*pi solid angle,Modelled,Relative position within this comparable cohor...
5,environmental_exposure_tradeoff,A3CLR22_3_Faustini_Rim_A:diagnostic_component_...,sky_visible_fraction_4pi,0.504919,lower,7,fraction of 4*pi solid angle,Modelled,Relative position within this comparable cohor...
6,environmental_exposure_tradeoff,A3CLR22_3_Faustini_Rim_A:diagnostic_component_...,sky_visible_fraction_4pi,0.499492,lower,2,fraction of 4*pi solid angle,Modelled,Relative position within this comparable cohor...
7,environmental_exposure_tradeoff,A3CLR22_3_Faustini_Rim_A:diagnostic_component_...,sky_visible_fraction_4pi,0.502659,lower,4,fraction of 4*pi solid angle,Modelled,Relative position within this comparable cohor...
8,environmental_exposure_tradeoff,A3CLR22_4_deGerlache_Rim_2:diagnostic_componen...,sky_visible_fraction_4pi,0.506145,lower,10,fraction of 4*pi solid angle,Modelled,Relative position within this comparable cohor...
9,environmental_exposure_tradeoff,A3CLR22_4_deGerlache_Rim_2:diagnostic_componen...,sky_visible_fraction_4pi,0.492688,lower,1,fraction of 4*pi solid angle,Modelled,Relative position within this comparable cohor...


,candidate_id,region,photoemission_opportunity_proxy,shadow_opportunity_proxy,psr_fraction_cells,surface_potential_V,status,reason,source_ids
0,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,A3CLR22_10_Leibnitz_Beta_Plateau,0.52364,0.47636,0.0,NaN,Unknown,Sunlit photoemission and dark/plasma-poor char...,CHARGING_MODEL; PSR_CHARGING
1,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,A3CLR22_10_Leibnitz_Beta_Plateau,0.54260,0.45740,0.0,NaN,Unknown,Sunlit photoemission and dark/plasma-poor char...,CHARGING_MODEL; PSR_CHARGING
2,A3CLR22_10_Leibnitz_Beta_Plateau:diagnostic_co...,A3CLR22_10_Leibnitz_Beta_Plateau,0.51422,0.48578,0.0,NaN,Unknown,Sunlit photoemission and dark/plasma-poor char...,CHARGING_MODEL; PSR_CHARGING
3,A3CLR22_11_Malapert_Massif:diagnostic_componen...,A3CLR22_11_Malapert_Massif,0.53466,0.46534,0.0,NaN,Unknown,Sunlit photoemission and dark/plasma-poor char...,CHARGING_MODEL; PSR_CHARGING
4,A3CLR22_11_Malapert_Massif:diagnostic_componen...,A3CLR22_11_Malapert_Massif,0.53088,0.46912,0.0,NaN,Unknown,Sunlit photoemission and dark/plasma-poor char...,CHARGING_MODEL; PSR_CHARGING


**[Open science comparison, reference assumptions and options](outputs/environment_comparison/20261109T184313460859Z_732b52/evidence_context/science/SCIENCE_COMPARISON.md)**

## Historical monthly and quarterly environmental context

The temporal extension uses actual downloaded UV/plasma observations and checked radiation summaries. UTC calendar periods are **not lunar seasons**. Coverage, spacecraft position, cadence and non-overlapping histories remain explicit; candidate scores and operational Unknowns are unchanged.

In [8]:
temporal = report.get("evidence_rules", {}).get("temporal_context")
if temporal:
    temporal_root = comparison_run / "evidence_context/sources/temporal"
    temporal_support = pd.read_csv(temporal_root / "temporal_support.csv")
    temporal_monthly = pd.read_csv(temporal_root / "monthly_context.csv")
    temporal_quarterly = pd.read_csv(temporal_root / "quarterly_context.csv")
    assert temporal["new_scored_factors"] == 0
    assert temporal["candidate_operational_conclusion"] == "Unknown"
    assert not temporal_monthly.loc[temporal_monthly.valid_days.eq(0), "median_of_daily_medians"].notna().any()
    assert temporal_quarterly.period_kind.eq("calendar_quarter").all()
    display(temporal_support[["instrument", "series_id", "start_utc", "stop_exclusive_utc", "valid_dates"]])
    display(temporal_monthly.head(12))
    display(temporal_quarterly.head(12))
    display(pd.read_csv(temporal_root / "cross_dataset_overlap.csv"))
    display(Markdown("**[Open historical time-period findings and coverage](" + (comparison_run / temporal["entry_point"]).relative_to(SOURCE).as_posix() + ")**"))
else:
    print("No temporal extension configured; no time-series evidence fabricated.")

,instrument,series_id,start_utc,stop_exclusive_utc,valid_dates
0,ARTEMIS_THB,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_negative_1...,2023-10-01T00:00:00+00:00,2026-09-30T00:00:00+00:00,1014
1,ARTEMIS_THB,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_negative_5...,2023-11-19T00:00:00+00:00,2026-09-30T00:00:00+00:00,758
2,ARTEMIS_THB,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_nonnegativ...,2023-10-01T00:00:00+00:00,2026-09-30T00:00:00+00:00,1014
3,ARTEMIS_THB,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_nonnegativ...,2023-10-01T00:00:00+00:00,2026-09-30T00:00:00+00:00,786
4,ARTEMIS_THB,PLASMA_THB_v01_thb_fgs_btotal_all_positions,2023-10-01T00:00:00+00:00,2026-09-30T00:00:00+00:00,1038
...,...,...,...,...,...
96,GOES19,RAD_GOES19_3-0-3_s1_c7_41000.0-74000.0keV,2026-01-10T00:00:00+00:00,2026-11-01T00:00:00+00:00,193
97,GOES19,RAD_GOES19_3-0-3_s1_c8_78000.0-99700.0keV,2026-01-10T00:00:00+00:00,2026-11-01T00:00:00+00:00,192
98,GOES19,RAD_GOES19_3-0-3_s1_c9_97900.0-120300.0keV,2026-01-10T00:00:00+00:00,2026-11-01T00:00:00+00:00,1
99,SORCE_SOLSTICE,UV_SOLSTICE_FUV_mode11_v18,2003-05-14T00:00:00+00:00,2020-02-26T00:00:00+00:00,5700


,series_id,period,year,period_kind,period_start_utc,period_stop_exclusive_utc,expected_calendar_days,recorded_days,valid_days,valid_day_fraction,...,valid_seconds,valid_time_fraction,median_of_daily_medians,p95_of_daily_medians,maximum_observed_value,maximum_daily_p95,comparison_scope,observed_level_order_within_year,comparable_nonempty_periods,coverage_note
0,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_negative_1...,2023-10,2023,calendar_month,2023-10-01T00:00:00+00:00,2023-11-01T00:00:00+00:00,31,28,28,0.903226,...,NaN,NaN,6.055,14.220500,25.879999,25.505001,historical same-series descriptive levels; NOT...,1.0,3,Check valid days/time and position/cadence; ra...
1,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_negative_1...,2023-11,2023,calendar_month,2023-11-01T00:00:00+00:00,2023-12-01T00:00:00+00:00,30,28,28,0.933333,...,NaN,NaN,6.185,13.751750,20.360001,19.552500,historical same-series descriptive levels; NOT...,2.0,3,Check valid days/time and position/cadence; ra...
2,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_negative_1...,2023-12,2023,calendar_month,2023-12-01T00:00:00+00:00,2024-01-01T00:00:00+00:00,31,31,31,1.000000,...,NaN,NaN,7.620,13.622500,26.370001,24.167500,historical same-series descriptive levels; NOT...,3.0,3,Check valid days/time and position/cadence; ra...
3,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_negative_1...,2024-01,2024,calendar_month,2024-01-01T00:00:00+00:00,2024-02-01T00:00:00+00:00,31,31,31,1.000000,...,NaN,NaN,5.860,10.322500,18.350000,17.944500,historical same-series descriptive levels; NOT...,2.0,12,Check valid days/time and position/cadence; ra...
4,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_negative_1...,2024-02,2024,calendar_month,2024-02-01T00:00:00+00:00,2024-03-01T00:00:00+00:00,29,21,21,0.724138,...,NaN,NaN,5.545,9.790000,17.730000,16.566500,historical same-series descriptive levels; NOT...,1.0,12,Check valid days/time and position/cadence; ra...
5,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_negative_1...,2024-03,2024,calendar_month,2024-03-01T00:00:00+00:00,2024-04-01T00:00:00+00:00,31,31,31,1.000000,...,NaN,NaN,5.910,14.915000,51.230000,49.122002,historical same-series descriptive levels; NOT...,3.0,12,Check valid days/time and position/cadence; ra...
6,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_negative_1...,2024-04,2024,calendar_month,2024-04-01T00:00:00+00:00,2024-05-01T00:00:00+00:00,30,30,30,1.000000,...,NaN,NaN,6.210,14.658750,26.690001,26.376999,historical same-series descriptive levels; NOT...,4.0,12,Check valid days/time and position/cadence; ra...
7,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_negative_1...,2024-05,2024,calendar_month,2024-05-01T00:00:00+00:00,2024-06-01T00:00:00+00:00,31,31,31,1.000000,...,NaN,NaN,7.260,16.097500,46.880001,42.810000,historical same-series descriptive levels; NOT...,8.0,12,Check valid days/time and position/cadence; ra...
8,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_negative_1...,2024-06,2024,calendar_month,2024-06-01T00:00:00+00:00,2024-07-01T00:00:00+00:00,30,28,28,0.933333,...,NaN,NaN,6.475,12.571500,22.870001,22.575000,historical same-series descriptive levels; NOT...,6.0,12,Check valid days/time and position/cadence; ra...
9,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_negative_1...,2024-07,2024,calendar_month,2024-07-01T00:00:00+00:00,2024-08-01T00:00:00+00:00,31,30,29,0.935484,...,NaN,NaN,6.390,15.519000,22.690001,22.611000,historical same-series descriptive levels; NOT...,5.0,12,Check valid days/time and position/cadence; ra...


,series_id,period,year,period_kind,period_start_utc,period_stop_exclusive_utc,expected_calendar_days,recorded_days,valid_days,valid_day_fraction,...,valid_seconds,valid_time_fraction,median_of_daily_medians,p95_of_daily_medians,maximum_observed_value,maximum_daily_p95,comparison_scope,observed_level_order_within_year,comparable_nonempty_periods,coverage_note
0,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_negative_1...,2023Q4,2023,calendar_quarter,2023-10-01T00:00:00+00:00,2024-01-01T00:00:00+00:00,92,87,87,0.945652,...,NaN,NaN,6.4650,14.245500,26.370001,25.505001,historical same-series descriptive levels; NOT...,1.0,1,Check valid days/time and position/cadence; ra...
1,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_negative_1...,2024Q1,2024,calendar_quarter,2024-01-01T00:00:00+00:00,2024-04-01T00:00:00+00:00,91,83,83,0.912088,...,NaN,NaN,5.8600,11.818500,51.230000,49.122002,historical same-series descriptive levels; NOT...,1.0,4,Check valid days/time and position/cadence; ra...
2,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_negative_1...,2024Q2,2024,calendar_quarter,2024-04-01T00:00:00+00:00,2024-07-01T00:00:00+00:00,91,89,89,0.978022,...,NaN,NaN,6.4250,15.784000,46.880001,42.810000,historical same-series descriptive levels; NOT...,2.0,4,Check valid days/time and position/cadence; ra...
3,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_negative_1...,2024Q3,2024,calendar_quarter,2024-07-01T00:00:00+00:00,2024-10-01T00:00:00+00:00,92,91,90,0.978261,...,NaN,NaN,8.1750,17.771000,33.680000,32.056000,historical same-series descriptive levels; NOT...,4.0,4,Check valid days/time and position/cadence; ra...
4,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_negative_1...,2024Q4,2024,calendar_quarter,2024-10-01T00:00:00+00:00,2025-01-01T00:00:00+00:00,92,91,91,0.989130,...,NaN,NaN,7.8550,15.240000,62.930000,60.212999,historical same-series descriptive levels; NOT...,3.0,4,Check valid days/time and position/cadence; ra...
5,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_negative_1...,2025Q1,2025,calendar_quarter,2025-01-01T00:00:00+00:00,2025-04-01T00:00:00+00:00,90,87,87,0.966667,...,NaN,NaN,7.7700,16.570500,33.669998,29.662000,historical same-series descriptive levels; NOT...,3.0,4,Check valid days/time and position/cadence; ra...
6,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_negative_1...,2025Q2,2025,calendar_quarter,2025-04-01T00:00:00+00:00,2025-07-01T00:00:00+00:00,91,91,90,0.989011,...,NaN,NaN,8.6850,18.364250,40.650002,38.852500,historical same-series descriptive levels; NOT...,4.0,4,Check valid days/time and position/cadence; ra...
7,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_negative_1...,2025Q3,2025,calendar_quarter,2025-07-01T00:00:00+00:00,2025-10-01T00:00:00+00:00,92,89,88,0.956522,...,NaN,NaN,7.4325,17.858500,30.650000,26.956999,historical same-series descriptive levels; NOT...,1.0,4,Check valid days/time and position/cadence; ra...
8,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_negative_1...,2025Q4,2025,calendar_quarter,2025-10-01T00:00:00+00:00,2026-01-01T00:00:00+00:00,92,88,88,0.956522,...,NaN,NaN,7.5800,15.665500,37.090000,28.852000,historical same-series descriptive levels; NOT...,2.0,4,Check valid days/time and position/cadence; ra...
9,PLASMA_THB_v01_thb_fgs_btotal_SSE_x_negative_1...,2026Q1,2026,calendar_quarter,2026-01-01T00:00:00+00:00,2026-04-01T00:00:00+00:00,90,87,86,0.955556,...,NaN,NaN,7.4625,18.381251,34.330002,33.735998,historical same-series descriptive levels; NOT...,3.0,3,Check valid days/time and position/cadence; ra...


,left,right,span_overlap_days,simultaneous_correlation_performed,reason
0,UV,plasma,0.0,False,Span overlap is not matched qualified timestam...
1,UV,radiation,0.0,False,Span overlap is not matched qualified timestam...
2,plasma,radiation,1095.0,False,Span overlap is not matched qualified timestam...


**[Open historical time-period findings and coverage](outputs/environment_comparison/20261109T184313460859Z_732b52/evidence_context/TEMPORAL_CONTEXT.md)**

In [9]:
RUN_SYNTHETIC_TEST = False
if RUN_SYNTHETIC_TEST:
    sample_run = engine.main(['--sample'])
    sample_check = pd.read_csv(sample_run/'sample_verification.csv')
    assert len(sample_check) == 10 and sample_check.matches_expected.all()
    display(sample_check)
else:
    print('Synthetic scenario skipped. The main results use real existing EDA only.')


Synthetic scenario skipped. The main results use real existing EDA only.


## 6. One place to open the current product

Start at [LATEST_RESULTS.md](outputs/LATEST_RESULTS.md). It links only to the newest successful real-data product, not old readiness exports or synthetic tests. Each run's `RUN_SUMMARY.md` explains the rankings, coverage and available thermal profiles.

No empty optional CSV is written. The output catalog records why an unavailable optional product was omitted. Historical runs are retained, not silently deleted.


In [10]:
catalog = pd.read_csv(comparison_run/'output_catalog.csv')
display(catalog[['file','title','rows','status','reason']])
for entry in catalog.to_dict('records'):
    path = comparison_run/entry['file']
    if entry['status'] == 'omitted_empty':
        assert not path.exists()
    else:
        frame = pd.read_csv(path)
        assert len(frame) > 0
        if not entry['file'].startswith('evidence_context/'):
            # Preserve the existing clean candidate-product contract.
            assert not frame.isna().any().any()
            assert not frame.astype(str).apply(lambda column: column.str.lower().eq('unknown').any()).any()
        # The separately labelled evidence annex preserves source N/A values and
        # explicit Unknown inference limits; never fill those to make a clean score.
assert (OUTPUTS/'LATEST_RESULTS.md').is_file()
assert json.loads((comparison_run/'provenance.json').read_text())['status'] == 'complete'
assert not (ROOT/'data').exists()
print('Open:', OUTPUTS/'LATEST_RESULTS.md')


,file,title,rows,status,reason
0,environmental_rankings.csv,Ranked candidates with complete configured env...,14,has_rows,Rows exported; inspect their status/scope befo...
1,environmental_top_k.csv,Top environmental comparisons,5,has_rows,Rows exported; inspect their status/scope befo...
2,candidate_status.csv,"Every candidate accounted for, including exclu...",21,has_rows,Rows exported; inspect their status/scope befo...
3,excluded_candidates.csv,Excluded or insufficient-evidence candidates a...,7,has_rows,Rows exported; inspect their status/scope befo...
4,soft_preference_results.csv,Actual per-factor score contributions for rank...,28,has_rows,Rows exported; inspect their status/scope befo...
...,...,...,...,...,...
70,evidence_context/science/science_operational_r...,Radiation/UV/plasma applicability and limit holds,63,has_rows,Rows exported; inspect their status/scope befo...
71,evidence_context/science/science_source_catalo...,Pinned science input tables and original sourc...,6,has_rows,Rows exported; inspect their status/scope befo...
72,evidence_context/science/science_model_checks.csv,"Model-contract handling checks, NOT site safety",3,has_rows,Rows exported; inspect their status/scope befo...
73,evidence_context/science/candidate_plasma_cont...,Illumination context and unqualified charging ...,21,has_rows,Rows exported; inspect their status/scope befo...


C:\Users\silen\AppData\Local\Temp\ipykernel_18400\1428774400.py:8: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  frame = pd.read_csv(path)


Open: C:\Darrel\Self-Projects\1 group proj\NASA-Space-Apps-Challenge-2026\darrel stuff\Pipeline_Proto\outputs\LATEST_RESULTS.md


## Working product and scope

The normal run now delivers populated real-data products: 14 ranked comparisons, seven explained exclusions, all 21 candidate statuses, 100 supported evidence records, complete thermal profiles where available, and factor/coverage explanations. Missing physical evidence was not filled, scored or silently passed.

This is the working environmental comparison layer. It is not a trained environmental CNN, a complete operational Stage 3 assessment, a CNN-patch association or a certified base-location recommendation. Optional software tests and operational checks remain separate.


Steps 4-5 now include the scoped rule/evidence register, actual applied-filter explanations and all-candidate limitation cards. The historical/annual/regional references have not changed candidate scores or supplied missing operational inputs. See `Pipeline_Proto/README.md` for the current evidence and completion audit.